# TP 2 — Explorer les espaces de couleurs
**IMN119 : Acquisition de médias numériques**  
**Travail individuel — 100 points**

Votre mission : calculer l’effet des réglages de couleurs et programmer les
conversions RVB → CIELAB et RVB → HSV. Les interfaces et les conversions inverses
sont fournies pour explorer les résultats.

### Organisation
1. Dans Colab, choisissez **Fichier → Enregistrer une copie dans Drive**, puis
   renommez votre copie `TP2_VOTRE_CIP.ipynb`.
2. Utilisez une session **Python 3 sur CPU**. Exécutez les deux cellules de préparation pour installer et importer les outils du TP.
3. Complétez les cellules `REPONSE_Q01` à `REPONSE_Q07`. Conservez les marqueurs,
   les noms des fonctions, leurs paramètres et les clés des dictionnaires.
4. Remplacez `None` par vos réponses numériques, booléennes ou vos codes de choix.
   Dans les deux fonctions, remplacez les `raise NotImplementedError` par votre code.
5. Avant la remise, redémarrez la session et exécutez les cellules dans l’ordre.
   Remettez le **notebook exécuté** et l’archive produite à la fin.

Les images d’essai sont chargées depuis des liens en ligne.
L’ajout d’une image personnelle est facultatif.

| Question | Objet | Points |
|---|---|---:|
| Q01 | Calculer les résultats de réglages RVB | 10 |
| Q02 | Programmer la conversion RVB → CIELAB | 30 |
| Q03 | Programmer la conversion RVB → HSV | 30 |
| Q04 | Distinguer représentation, domaine et gamut | 5 |
| Q05 | Associer les manipulations aux espaces de couleurs | 5 |
| Q06 | Reconnaître les avantages et limites de RVB et HSV | 5 |
| Q07 | Calculer l’effet des neuf curseurs | 15 |
| **Total** | | **100** |

Toutes les questions seront évaluées automatiquement. Les essais fournis pour les
fonctions servent à progresser; ils ne couvrent pas tous les cas de la correction.
Respectez les formats indiqués. Les résultats numériques seront comparés avec une
tolérance absolue de `1e-6`, sauf indication différente; utilisez des valeurs non
arrondies ou au moins six chiffres après la virgule. Écrivez les décimales avec un point.


## Identification
Entrez votre nom et votre **CIP de l’Université de Sherbrooke**.
Votre CIP servira à nommer le notebook et l’archive de remise.


In [ ]:
# REPONSE_IDENTITE
nom = ""
cip = ""  # Votre CIP, en minuscules, sans espace.


## Préparation — installer et importer les outils
Exécutez **Installer les outils du TP2**, puis la cellule d’importation. `pip`
installe le paquet du cours depuis son dépôt avec les dépendances nécessaires.
À chaque nouvelle session Colab, exécutez ces deux cellules dans le même ordre.

Les fonctions de conversion que vous programmez restent dans les cellules de réponse.
Les images sont des tableaux NumPy de forme `(hauteur, largeur, 3)`. Le dernier indice
désigne le canal. Rouge correspond à `0`, vert à `1` et bleu à `2`.
Les fonctions à programmer reçoivent des **flottants entre 0 et 1**.


In [ ]:
#@title Installer les outils du TP2
%pip install -q "imn119-tp2 @ git+https://github.com/kyleross-dev/imn119-tp2.git@v2.1.2"


In [ ]:
import numpy as np
from imn119_tp2 import outils

outils.preparer()


### Images en ligne — code fourni
Exécutez la cellule ci-dessous pour charger les deux images depuis Wikimedia Commons.
Une connexion Internet est nécessaire. Vous pouvez consulter les sources :

- [Roue chromatique](https://commons.wikimedia.org/wiki/File:Color_circle_(RGB).png)
  — Crossover1370 et LaundryPizza03, [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/).
- [Spectre visible](https://commons.wikimedia.org/wiki/File:Linear_visible_spectrum_ciecam02.png)
  — SiriusB, [CC0](https://creativecommons.org/publicdomain/zero/1.0/).

Les images sont réduites en conservant leurs proportions pour accélérer les éditeurs.


In [ ]:
images = outils.charger_images()


### Ajouter une image personnelle — facultatif
Pour essayer une autre image, passez `importer_une_image` à `True` et exécutez
la cellule. Dans Colab, sélectionnez un fichier PNG ou JPEG sur votre ordinateur.
Il sera ajouté au menu des éditeurs; relancez leur cellule pour actualiser le menu.
Une grande image est réduite en conservant ses proportions pour accélérer les interactions.


In [ ]:
importer_une_image = False
if importer_une_image:
    outils.importer_images(images)


### Utiliser les outils d’affichage
Les outils d’affichage et les curseurs sont disponibles après la préparation.
Chaque éditeur montre l’original, le résultat et les trois canaux du résultat.
Les canaux RVB sont affichés en couleur; les canaux CIELAB et HSV en niveaux de gris,
sur des échelles fixes. Pour CIELAB, l’échelle d’affichage usuelle de a* et b* va de
−128 à 127; elle ne définit pas une borne mathématique universelle.

Si les curseurs ne s’affichent pas, réexécutez la préparation puis la cellule de
l’éditeur. Vous pouvez aussi utiliser les **essais sans widgets** sous les questions 1 à 3 :
modifiez les valeurs numériques et relancez la cellule.


## Question 1 — Calculer les réglages RVB
Lisez la fonction fournie `appliquer_rgb`, puis calculez les résultats ci-dessous.
Chaque facteur multiplie le canal correspondant; les valeurs obtenues sont ensuite
bornées dans `[0, 1]`. Le facteur `1` conserve un canal et `0` l’annule.

On considère le pixel **(R, V, B) = (0.8, 0.4, 0.2)**.

| Clé à compléter | Facteurs (rouge, vert, bleu) |
|---|---|
| `neutre` | `(1, 1, 1)` |
| `reglage` | `(1.5, 0.5, 2)` |
| `tout_zero` | `(0, 0, 0)` |

Pour chaque cas, remplacez `None` par un triplet de nombres dans l’ordre **R, V, B**,
par exemple `(0.1, 0.2, 0.3)`. Vous pouvez écrire des expressions Python.
Utilisez ensuite l’éditeur pour comparer les effets sur la roue et le spectre.


In [ ]:
def appliquer_rgb(tableau, rouge=1.0, vert=1.0, bleu=1.0):
    """Multiplie les canaux RVB, puis borne le résultat entre 0 et 1."""
    resultat = np.array(tableau, dtype=float, copy=True)
    resultat[..., 0] *= rouge
    resultat[..., 1] *= vert
    resultat[..., 2] *= bleu
    return np.clip(resultat, 0, 1)


In [ ]:
# REPONSE_Q01
q01 = {
    "neutre": None,
    "reglage": None,
    "tout_zero": None,
}


In [ ]:
interface_rvb = outils.creer_editeur(images, "RVB")


### Essai sans widgets — RVB
Modifiez les trois facteurs, puis relancez la cellule.


In [ ]:
outils.afficher_comparaison(images, "RVB", image="Roue chromatique", reglages=(1.0, 0.5, 1.0))


## Question 2 — Programmer la conversion RVB → CIELAB
Complétez **uniquement** `rgb_to_lab(tableau)` selon l’algorithme A ci-dessous.
Votre code doit traiter tous les pixels et renvoyer un tableau flottant de même forme,
sans modifier le tableau d’entrée. Conservez la signature de la fonction pour les essais automatiques.

Vous devez écrire la conversion : n’utilisez pas une fonction de bibliothèque
telle que `rgb2lab` pour produire le résultat. La conversion inverse `lab2rgb` est fournie.

### Algorithme A
**Entrée :** sRVB normalisé dans `[0, 1]`.

**1. Linéariser les canaux.** Appliquez la même règle à chaque canal `C` parmi `R`, `G` et `B`.
Si `C > 0.04045`, calculez :

$$
C_{lin} = \left(\frac{C + 0.055}{1.055}\right)^{2.4}
$$

Sinon, calculez :

$$
C_{lin} = \frac{C}{12.92}
$$

Multipliez ensuite les trois canaux linéarisés par **100**. Dans les équations
suivantes, `R`, `G` et `B` désignent ces valeurs linéarisées et multipliées par 100.

**2. Calculer les coordonnées XYZ.**

$$
X = 0.4124R + 0.3576G + 0.1805B
$$

$$
Y = 0.2126R + 0.7152G + 0.0722B
$$

$$
Z = 0.0193R + 0.1192G + 0.9505B
$$

**3. Normaliser avec le blanc de référence D65.**

$$
X_n = \frac{X}{95.047}
$$

$$
Y_n = \frac{Y}{100}
$$

$$
Z_n = \frac{Z}{108.883}
$$

**4. Appliquer la fonction de transformation.** Pour chacun des tableaux
normalisés `T` parmi `X_n`, `Y_n` et `Z_n`, si `T > 0.008856`, calculez :

$$
f(T) = T^{1/3}
$$

Sinon, calculez :

$$
f(T) = 7.787T + \frac{16}{116}
$$

**5. Calculer les canaux CIELAB.**

$$
L^{*} = 116f(Y_n) - 16
$$

$$
a^{*} = 500\left[f(X_n) - f(Y_n)\right]
$$

$$
b^{*} = 200\left[f(Y_n) - f(Z_n)\right]
$$

**Sortie :** `np.stack((l, a, b), axis=-1)`. L\* varie approximativement de 0 à 100;
a\* et b\* sont des coordonnées signées. Ne normalisez pas ces trois canaux dans `[0, 1]`.
Ne faites pas d’écrêtage dans votre fonction de conversion.

**Conseils NumPy.** `tableau[..., 0]` sélectionne le rouge de tous les pixels.
`np.where(condition, valeur_si_vrai, valeur_si_faux)` applique un choix pixel par pixel.
En Python, une puissance s’écrit `**`, et non `^`.


In [ ]:
# REPONSE_Q02
def rgb_to_lab(tableau):
    """sRVB [0,1] -> CIELAB D65, selon l’algorithme A.

    Entrée : tableau flottant (hauteur, largeur, 3).
    Sortie : même forme, canaux L*, a*, b*, sans modifier l’entrée.
    """
    # ----------- CODE À COMPLÉTER -----------
    raise NotImplementedError("Complétez rgb_to_lab selon l’algorithme A.")
    # ----------------- FIN -----------------


### Essais fournis — CIELAB
Exécutez la fonction, puis cette cellule. Les valeurs attendues sont arrondies;
la tolérance tient compte des coefficients du document. Les essais vérifient
quelques couleurs, la forme du résultat et la préservation de l’entrée.


In [ ]:
outils.tester_lab(rgb_to_lab)


### Explorer vos résultats — CIELAB
Après avoir complété votre fonction, lancez l’éditeur. Les trois curseurs ajoutent
un décalage aux canaux. Le réglage neutre est **(0, 0, 0)**.
Modifiez L\*, a\* et b\* séparément et observez les différences avec RVB.
Le retour CIELAB → sRVB peut écrêter les couleurs hors du gamut d’affichage.

Après toute modification de votre fonction, relancez cette cellule pour utiliser la nouvelle version.


In [ ]:
interface_lab = outils.creer_editeur(images, "CIELAB", conversion=rgb_to_lab)


### Essai sans widgets — CIELAB
Les trois valeurs sont des décalages : L\*, a\*, b\*.


In [ ]:
outils.afficher_comparaison(images, "CIELAB", image="Roue chromatique", reglages=(15, 0, 0), conversion=rgb_to_lab)


## Question 3 — Programmer la conversion RVB → HSV
Complétez **uniquement** `rgb_to_hsv(tableau)` selon l’algorithme B.
Renvoyez un tableau flottant de même forme, sans modifier l’entrée.
N’utilisez pas une conversion de bibliothèque telle que `rgb2hsv` pour obtenir votre résultat.
La conversion inverse `hsv2rgb` est fournie.

### Algorithme B
**Entrée :** R, G et B normalisés dans `[0, 1]`.

**1. Calculer les extrema par pixel.** Calculez le maximum et le minimum
des trois canaux pour chaque pixel, et non sur l’image entière.

$$
M = \max(R, G, B)
$$

$$
m = \min(R, G, B)
$$

**2. Calculer la valeur et la saturation.**

$$
V = M
$$

Si `M > 0`, calculez :

$$
S = 1 - \frac{m}{M}
$$

Si `M = 0`, prenez **S = 0**.

**3. Calculer l’angle pour les pixels colorés.**

$$
D = \sqrt{R^2 + G^2 + B^2 - RG - RB - GB}
$$

$$
\theta = \arccos\left(\frac{R - 0.5G - 0.5B}{D}\right)
$$

`np.arccos` renvoie des radians. Convertissez cet angle **en degrés** avec
`np.degrees`, et nommez le résultat `theta_deg`.

**4. Déterminer et normaliser la teinte.**

- Si `G >= B`, prenez `H_deg = theta_deg`.
- Si `B > G`, prenez `H_deg = 360 - theta_deg`.

Normalisez ensuite la teinte en divisant `H_deg` par 360 et en appliquant le modulo 1 :

```python
h = (h_deg / 360) % 1
```

**Cas particuliers et stabilité numérique :**

- Pour un pixel gris, blanc ou noir (`R = G = B`), prenez **H = 0**.
- Évitez toute division par zéro dans le calcul de S et de l’angle. Vous pouvez utiliser
  `np.divide(..., out=..., where=...)` ou un dénominateur temporaire non nul.
- Arrondis : bornez l’expression sous la racine à au moins 0 et l’argument de
  `np.arccos` dans `[-1, 1]` avec `np.clip`.
- `np.where` évalue ses deux branches; remplacer simplement une division par zéro
  dans une branche ne suffit donc pas à l’éviter.

**Sortie :** `np.stack((h, s, v), axis=-1)`. Les **trois canaux sont normalisés
dans `[0, 1]`**, y compris la teinte. Seul le décalage du curseur est exprimé en degrés.

**Convention retenue.** La formule de teinte par arccos mesure un angle dans le plan
chromatique et diffère légèrement de la teinte HSV calculée par secteurs dans les
bibliothèques. Un aller-retour avec `hsv2rgb` peut donc modifier légèrement certaines
couleurs même au réglage neutre. Les essais suivent **l’algorithme B**; une égalité
exacte avec `rgb2hsv` n’est pas demandée.


In [ ]:
# REPONSE_Q03
def rgb_to_hsv(tableau):
    """RVB [0,1] -> HSV [0,1], selon l’algorithme B (teinte par arccos).

    Entrée : tableau flottant (hauteur, largeur, 3).
    Sortie : même forme, canaux H, S, V; H=0 pour les pixels achromatiques.
    """
    # ----------- CODE À COMPLÉTER -----------
    raise NotImplementedError("Complétez rgb_to_hsv selon l’algorithme B.")
    # ----------------- FIN -----------------


### Essais fournis — HSV
Les teintes du rouge, jaune, vert, cyan, bleu et magenta valent respectivement
0, 1/6, 2/6, 3/6, 4/6 et 5/6. Un cas supplémentaire distingue la formule par arccos
de la conversion usuelle par secteurs. Les pixels gris doivent rester finis.


In [ ]:
outils.tester_hsv(rgb_to_hsv)


### Explorer vos résultats — HSV
Le réglage neutre est **(H + 0°, S × 1, V × 1)**.
La teinte est décalée et revient au début après un tour; saturation et valeur sont
multipliées puis bornées dans `[0, 1]`. Essayez notamment S × 0, V × 0 et H + 120°.
Vous utiliserez les mêmes règles de calcul à la question 7.

Après toute modification de votre fonction, relancez cette cellule pour utiliser la nouvelle version.


In [ ]:
interface_hsv = outils.creer_editeur(images, "HSV", conversion=rgb_to_hsv)


### Essai sans widgets — HSV
Valeurs : décalage de teinte en degrés, facteur de saturation, facteur de valeur.


In [ ]:
outils.afficher_comparaison(images, "HSV", image="Roue chromatique", reglages=(120, 1, 1), conversion=rgb_to_hsv)


## Question 4 — Représentation et gamut
Pour chaque affirmation, remplacez `None` par **`True`** ou **`False`**.
N’utilisez pas les chaînes de caractères `"True"` ou `"False"`.
On considère des images sRVB normalisées et les conventions de ce TP.

| Clé | Affirmation |
|---|---|
| `srvb_vers_lab` | Tout pixel sRVB dans `[0, 1]³` peut être converti en CIELAB avec l’algorithme A. |
| `lab_toujours_affichable` | Tout triplet CIELAB réel, sans restriction de valeurs, correspond à une couleur affichable sans écrêtage sur un écran sRVB. |
| `teinte_gris_unique` | Un pixel gris possède une teinte physique unique qui reste définie quand sa saturation vaut zéro. |
| `hdr_dans_domaine` | Un tableau RVB contenant la valeur `2.0` respecte le domaine d’entrée indiqué pour les fonctions de conversion du TP. |
| `retour_lab_ecretage` | Après modification de ses canaux, un pixel CIELAB peut nécessiter un écrêtage lors de son retour vers sRVB pour l’affichage. |


In [ ]:
# REPONSE_Q04
q04 = {
    "srvb_vers_lab": None,
    "lab_toujours_affichable": None,
    "teinte_gris_unique": None,
    "hdr_dans_domaine": None,
    "retour_lab_ecretage": None,
}


## Question 5 — Associer une manipulation à un espace
Pour chaque opération, indiquez **l’espace dont les canaux nommés permettent
directement la manipulation décrite**, en utilisant un seul code :
`"RVB"`, `"HSV"` ou `"CIELAB"`. Un code peut être réutilisé.

| Clé | Opération |
|---|---|
| `contribution_rouge` | Multiplier uniquement la contribution rouge R d’un pixel. |
| `decaler_teinte` | Ajouter un angle au canal de teinte H en conservant S et V. |
| `modifier_clarte` | Ajouter un décalage à la clarté L* en conservant a* et b*. |
| `reduire_saturation` | Multiplier le canal S par `0.5` en conservant H et V. |
| `axe_vert_rouge` | Décaler la coordonnée a* sur l’axe vert–rouge en conservant L* et b*. |


In [ ]:
# REPONSE_Q05
q05 = {
    "contribution_rouge": None,
    "decaler_teinte": None,
    "modifier_clarte": None,
    "reduire_saturation": None,
    "axe_vert_rouge": None,
}


## Question 6 — Avantages et limites
Associez chaque clé à **un seul numéro** parmi les six propositions ci-dessous.
Deux propositions sont des distracteurs. Utilisez des entiers, pas des textes.

| Numéro | Proposition |
|---:|---|
| 1 | La séparation de H, S et V facilite les réglages directs de teinte et de saturation. |
| 2 | Les canaux correspondent aux contributions des trois primaires de la synthèse additive d’un écran. |
| 3 | Des distances numériques égales entre des triplets H, S, V ne garantissent pas des écarts perceptifs égaux. |
| 4 | La clarté perceptuelle et la teinte ne sont pas isolées dans des canaux indépendants R, V et B. |
| 5 | Ce modèle permet à tout écran d’afficher toutes les couleurs perceptibles par l’humain. |
| 6 | Dans ce modèle, chaque couleur possède trois composantes égales par définition. |

Complétez `avantage_rvb`, `limite_rvb`, `avantage_hsv` et `limite_hsv`.


In [ ]:
# REPONSE_Q06
q06 = {
    "avantage_rvb": None,
    "limite_rvb": None,
    "avantage_hsv": None,
    "limite_hsv": None,
}


## Question 7 — Calculer l’effet des neuf curseurs
Calculez la **valeur finale du canal indiqué**, après réglage et écrêtage.
Les autres canaux n’interviennent pas dans le calcul demandé. Il ne faut pas
reconvertir le pixel dans un autre espace de couleurs.

Chaque cas d’augmentation et de diminution repart **indépendamment de la même
valeur initiale**. Les deux réglages ne s’appliquent pas successivement.

| Canal / clé | Valeur initiale | Réglage `augmenter` | Réglage `diminuer` | Domaine final |
|---|---:|---|---|---|
| Rouge `rouge` | `0.8` | × `1.5` | × `0.5` | `[0, 1]` |
| Vert `vert` | `0.4` | × `1.5` | × `0.5` | `[0, 1]` |
| Bleu `bleu` | `0.2` | × `1.5` | × `0.5` | `[0, 1]` |
| Clarté `L` | `60` | + `50` | − `50` | `[0, 100]` |
| Coordonnée `a` | `-110` | + `50` | − `50` | `[-128, 127]` |
| Coordonnée `b` | `110` | + `50` | − `50` | `[-128, 127]` |
| Teinte `H` | `0.9` | + `90°` | − `90°` | `[0, 1[`, retour cyclique |
| Saturation `S` | `0.7` | × `1.5` | × `0.5` | `[0, 1]` |
| Valeur `V` | `0.6` | × `1.5` | × `0.5` | `[0, 1]` |

Pour H, renvoyez une **teinte normalisée**, pas un angle en degrés.
Remplacez chaque `None` par un nombre ou une expression Python.


In [ ]:
# REPONSE_Q07
q07 = {
    "rouge": {"augmenter": None, "diminuer": None},
    "vert": {"augmenter": None, "diminuer": None},
    "bleu": {"augmenter": None, "diminuer": None},
    "L": {"augmenter": None, "diminuer": None},
    "a": {"augmenter": None, "diminuer": None},
    "b": {"augmenter": None, "diminuer": None},
    "H": {"augmenter": None, "diminuer": None},
    "S": {"augmenter": None, "diminuer": None},
    "V": {"augmenter": None, "diminuer": None},
}


## Vérification et remise
1. Complétez les sept questions et votre identification.
2. Redémarrez la session et exécutez toutes les cellules dans l’ordre.
3. Exécutez les deux cellules ci-dessous pour créer et télécharger votre archive.
4. Téléchargez aussi le notebook exécuté par **Fichier → Télécharger → Télécharger
   le fichier .ipynb**. Remettez le notebook et l’archive selon les consignes du cours.

L’archive contient les réponses structurées dans `reponses.json`, les deux fonctions
dans `conversions.py` et toutes vos cellules de réponse dans `reponses.py`.
Elle permettra la correction automatique ultérieure. L’export ne calcule aucune note
et accepte une remise partielle : les réponses manquantes restent identifiables.
Il faut avoir exécuté les sept cellules de réponse, même si certaines sont incomplètes.
Les fichiers de la session Colab sont temporaires.



In [ ]:
reponses = {"q01": q01, "q04": q04, "q05": q05, "q06": q06, "q07": q07}
archive = outils.creer_remise(nom, cip, reponses)


In [ ]:
outils.telecharger(archive)


### Références et provenance
Adaptation du matériel « TP3 — Acquisition des médias numériques » fourni dans
l’archive du cours (énoncé et trois scripts). Les images d’essai sont chargées
depuis Wikimedia Commons; leurs sources sont indiquées dans la préparation. Les deux algorithmes sont conservés. Les questions sont adaptées en calculs, fonctions
et choix structurés; les interfaces sont adaptées à Colab. Les consignes de remise suivent l’organisation du TP1 fourni en exemple.

- [TP1 — exemple d’organisation](https://colab.research.google.com/drive/1ibChFEeaXYn419l2_el4g8SOvOpYfcFq)
- [NumPy : where](https://numpy.org/doc/stable/reference/generated/numpy.where.html),
  [clip](https://numpy.org/doc/stable/reference/generated/numpy.clip.html),
  [divide](https://numpy.org/doc/stable/reference/generated/numpy.divide.html)
- [scikit-image : conversions de couleurs](https://scikit-image.org/docs/0.25.x/api/skimage.color.html)
- [Widgets dans Colab](https://github.com/googlecolab/colabtools/blob/main/google/colab/output/_widgets.py)

- [ipywidgets — interfaces interactives](https://ipywidgets.readthedocs.io/en/latest/examples/Using%20Interact.html)
